In [ ]:
import os
import json
import copy
import math
import random
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from joblib import dump, load

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from IPython.display import display
from IPython import get_ipython

from sklearn.decomposition import PCA as CPU_PCA

from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler

# GPU acceleration for supported sklearn estimators
USE_GPU_ACCEL = False
GPU_ACCEL_ACTIVE = False
CUML_ACCEL_AVAILABLE = False

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Seeds
BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]

SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)


def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


warnings.filterwarnings("ignore", category=UserWarning)

# Paths
ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"
IN_META = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 5"
OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_CACHE = ARTIFACTS / "cache" / NOTEBOOK_SUBDIR

for d in [OUT_REPORTS, OUT_META, OUT_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

# Device
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {TORCH_DEVICE}")

# Load backbone lock
LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(
        f"proteomics_backbone_lock.json not found at {LOCK_PATH}. "
        "Run the lock cell at the end of Notebook 3b first."
    )
with LOCK_PATH.open("r") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM = backbone_lock["track2_stress_test_arm"]

print("Backbone lock loaded.")
print(f"  Primary arm  : {PRIMARY_ARM}")
print(f"  Secondary arm: {SECONDARY_ARM}")
print(f"  Track 2 arm  : {TRACK2_ARM}")
print(f"  Seeds to run : {ALL_SEEDS}")

# Exact ranked configurations inherited from Notebook 4
TOP10_CONFIGS = [
    {"rank": 1,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna"},
    {"rank": 2,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+prot"},
    {"rank": 3,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna"},
    {"rank": 4,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+mut"},
    {"rank": 5,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 6,  "arm": "prot_combined_union",      "model": "elasticnet", "feature_set": "rna+prot"},
    {"rank": 7,  "arm": "prot_procan_depmapSanger", "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 8,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+mut"},
    {"rank": 9,  "arm": "prot_rppa_ccle",           "model": "elasticnet", "feature_set": "rna+mut+prot"},
    {"rank": 10, "arm": "prot_procan_depmapSanger", "model": "elasticnet", "feature_set": "prot"},
]

ACTIVE_ARMS = sorted({cfg["arm"] for cfg in TOP10_CONFIGS})
CONFIGS_BY_ARM = {
    arm: [cfg for cfg in TOP10_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

CNV_TEST_CONFIGS = (
    [{"rank": 1000, "arm": arm, "model": "elasticnet", "feature_set": "cnv"} for arm in ACTIVE_ARMS] +
    [{"rank": 1001, "arm": arm, "model": "elasticnet", "feature_set": "rna+cnv"} for arm in ACTIVE_ARMS] +
    [{"rank": 1002, "arm": arm, "model": "elasticnet", "feature_set": "cnv+mut"} for arm in ACTIVE_ARMS] +
    [{"rank": 1003, "arm": arm, "model": "elasticnet", "feature_set": "rna+cnv+mut"} for arm in ACTIVE_ARMS] +
    [{"rank": 1004, "arm": arm, "model": "elasticnet", "feature_set": "cnv+prot"} for arm in ACTIVE_ARMS] +
    [{"rank": 1005, "arm": arm, "model": "elasticnet", "feature_set": "rna+cnv+prot"} for arm in ACTIVE_ARMS] +
    [{"rank": 1006, "arm": arm, "model": "elasticnet", "feature_set": "rna+cnv+mut+prot"} for arm in ACTIVE_ARMS]
)

CNV_CONFIGS_BY_ARM = {
    arm: [cfg for cfg in CNV_TEST_CONFIGS if cfg["arm"] == arm]
    for arm in ACTIVE_ARMS
}

print("\nTop-10 ranked configs loaded.")
display(pd.DataFrame(TOP10_CONFIGS))
print("Active arms from top-10 configs:", ACTIVE_ARMS)

Torch device: cuda
Backbone lock loaded.
  Primary arm  : prot_procan_depmapSanger
  Secondary arm: prot_ms_ccle_gygi
  Track 2 arm  : prot_combined_union
  Seeds to run : [19537, 1584678, 17052356]

Top-10 ranked configs loaded.


,rank,arm,model,feature_set
0,1,prot_combined_union,elasticnet,rna
1,2,prot_rppa_ccle,elasticnet,rna+prot
2,3,prot_rppa_ccle,elasticnet,rna
3,4,prot_combined_union,elasticnet,rna+mut
4,5,prot_combined_union,elasticnet,rna+mut+prot
5,6,prot_combined_union,elasticnet,rna+prot
6,7,prot_procan_depmapSanger,elasticnet,rna+mut+prot
7,8,prot_rppa_ccle,elasticnet,rna+mut
8,9,prot_rppa_ccle,elasticnet,rna+mut+prot
9,10,prot_procan_depmapSanger,elasticnet,prot


Active arms from top-10 configs: ['prot_combined_union', 'prot_procan_depmapSanger', 'prot_rppa_ccle']


In [2]:
# Settings
N_DRUGS_BAKEOFF = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED = 10
RIDGE_ALPHA = 1.0
EN_ALPHA = 0.05
EN_L1_RATIO = 0.2
PCA_COMPONENTS = 100
PRIMARY_TARGET = "auc"

# Deep imputation settings
DEEP_EPOCHS = 200
DEEP_PATIENCE = 20
DEEP_BATCH_SIZE = 64
DEEP_LR = 1e-3
DEEP_WEIGHT_DECAY = 1e-5
DEEP_HIDDEN_FRAC = 0.5
DEEP_MIN_HIDDEN = 64
DEEP_MAX_HIDDEN = 1024
DEEP_NOISE_STD = 0.05
DEEP_DROPOUT = 0.1
VAL_FRAC = 0.15
MIN_VAL_SAMPLES = 24

CACHE_VERSION = "v1_deep"
CACHE_TAG = f"{CACHE_VERSION}__pca{PCA_COMPONENTS}__epochs{DEEP_EPOCHS}"

PROT_DEEP_LOCK_FILENAME = "deep_imputer_choice_prot.json"
CNV_DEEP_LOCK_FILENAME = "deep_imputer_choice_cnv.json"


# Helpers

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)


def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df


def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])


def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"


def safe_group_splits(
    cells: List[str],
    groups: pd.Series,
    n_splits_desired: int,
    seed: int,
) -> Tuple[List[Tuple[np.ndarray, np.ndarray]], str]:
    groups = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups = groups.nunique()
    n_cells = len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        sp = GroupKFold(n_splits=n_splits)
        return (
            list(sp.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)),
            f"GroupKFold(n_splits={n_splits})",
        )
    n_splits = min(max(2, n_splits), n_cells)
    sp = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(sp.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits})"


def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    if feature_set == "":
        return tuple()
    return tuple(feature_set.split("+"))


def concat_selected_modalities(
    mats: Dict[str, np.ndarray],
    selected_keys: Tuple[str, ...],
    n_rows: int,
) -> np.ndarray:
    parts = []
    for k in selected_keys:
        arr = mats.get(k)
        if arr is None or arr.shape[1] == 0:
            return np.zeros((n_rows, 0), dtype=np.float32)
        parts.append(arr)
    if not parts:
        return np.zeros((n_rows, 0), dtype=np.float32)
    return np.concatenate(parts, axis=1)


def make_model(model_name: str, seed: int):
    model_name = str(model_name).lower()
    if model_name == "ridge":
        return Ridge(alpha=RIDGE_ALPHA, random_state=seed)
    if model_name == "elasticnet":
        return ElasticNet(
            alpha=EN_ALPHA,
            l1_ratio=EN_L1_RATIO,
            random_state=seed,
            max_iter=10000,
        )
    raise ValueError(f"Unsupported model for Notebook 5 deep imputation: {model_name}")


def _safe_name(s: str) -> str:
    s = str(s)
    return (
        s.replace(os.sep, "_")
         .replace(" ", "_")
         .replace("+", "plus")
         .replace(":", "_")
         .replace("/", "_")
         .replace("\\", "_")
    )

def _base_cache_path(seed: int, arm: str, fold_i: int) -> Path:
    return OUT_CACHE / (
        f"base__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}.joblib"
    )


def load_or_build_base_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    train_cells: List[str],
    eligible_cells: List[str],
) -> dict:
    path = _base_cache_path(seed, arm, fold_i)
    if path.exists():
        return load(path)

    Xr = passthrough_transform_block(
        X_train=rna.loc[train_cells].to_numpy(dtype=float),
        X_all=rna.loc[eligible_cells].to_numpy(dtype=float),
        random_state=seed + 0,
        n_pca_components=PCA_COMPONENTS,
    )

    Xc = passthrough_transform_block(
        X_train=cnv.loc[train_cells].to_numpy(dtype=float),
        X_all=cnv.loc[eligible_cells].to_numpy(dtype=float),
        random_state=seed + 1,
        n_pca_components=PCA_COMPONENTS,
    )

    Xm = passthrough_transform_block(
        X_train=mut.loc[train_cells].to_numpy(dtype=float),
        X_all=mut.loc[eligible_cells].to_numpy(dtype=float),
        random_state=seed + 2,
        n_pca_components=PCA_COMPONENTS,
    )

    payload = {
        "Xr": Xr,
        "Xc": Xc,
        "Xm": Xm,
    }

    dump(payload, path)
    return payload

In [3]:
# Data load
cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")

rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))

cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col = pick_group_column(cell_index)
groups_all = (
    cell_index.set_index("depmap_id")[group_col]
    .astype("string").fillna("Unknown").astype(str)
)

core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
print(f"\nCore cohort (RNA ∩ CNV ∩ MUT): {len(core_cells)} cell lines")
print(f"Group column for CV: {group_col}")

prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()

prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][
    ["depmap_id", "compound_id", "y"]
].copy()

prot_arm_data: Dict[str, pd.DataFrame] = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} not found at {p}, skipping.")

drug_cov = (
    prism_auc.groupby("compound_id")["depmap_id"]
    .nunique().sort_values(ascending=False)
)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print(f"\nSelected {len(selected_drugs)} drugs for deep-imputation bake-off by PRISM AUC coverage.")


# Missingness report reused for deep notebook
print("MISSINGNESS REPORT")

avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()

availability = pd.DataFrame(avail_rows, index=core_cells)
availability.index.name = "depmap_id"
avail_path = OUT_REPORTS / "modality_availability_matrix.csv"
availability.to_csv(avail_path)
print(f"Availability matrix: {avail_path}  shape={availability.shape}")

avail_summary = (
    availability.sum().rename("n_cells_present").to_frame()
    .assign(
        n_cells_total=len(core_cells),
        pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2),
    )
)
avail_summary_path = OUT_REPORTS / "modality_availability_summary.csv"
avail_summary.to_csv(avail_summary_path)
print("\nModality availability summary:")
display(avail_summary)

feat_miss_rows = []
for arm, df in prot_arm_data.items():
    df_core = df.reindex(core_cells)
    col_miss = df_core.isna().mean()
    row_miss = df_core.isna().mean(axis=1)
    feat_miss_rows.append({
        "arm": arm,
        "n_features": int(df_core.shape[1]),
        "n_cells_in_core": int(df_core.shape[0]),
        "overall_missing_pct": float(df_core.isna().mean().mean() * 100),
        "col_miss_q10": float(col_miss.quantile(0.10)),
        "col_miss_q25": float(col_miss.quantile(0.25)),
        "col_miss_q50": float(col_miss.quantile(0.50)),
        "col_miss_q75": float(col_miss.quantile(0.75)),
        "col_miss_q90": float(col_miss.quantile(0.90)),
        "col_miss_q99": float(col_miss.quantile(0.99)),
        "row_miss_q10": float(row_miss.quantile(0.10)),
        "row_miss_q50": float(row_miss.quantile(0.50)),
        "row_miss_q90": float(row_miss.quantile(0.90)),
        "n_cols_fully_missing": int((col_miss == 1.0).sum()),
        "n_rows_fully_missing": int((row_miss == 1.0).sum()),
        "n_cols_zero_missing": int((col_miss == 0.0).sum()),
    })
feat_miss_df = pd.DataFrame(feat_miss_rows)
feat_miss_path = OUT_REPORTS / "per_arm_feature_missingness.csv"
feat_miss_df.to_csv(feat_miss_path, index=False)
print("\nPer-arm feature missingness:")
display(feat_miss_df)

pat_counts = None
platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0

    def pattern_label(row) -> str:
        parts = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(parts) if parts else "none"

    pat_counts = (
        plat_pres.apply(pattern_label, axis=1).rename("pattern")
        .value_counts().rename_axis("pattern").reset_index(name="n_cells")
    )
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    print("\nCombined union platform patterns:")
    display(pat_counts)

    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols:
            continue
        n_absent = int(plat_pres[key].eq(0).sum())
        miss_from_abs = n_absent * len(cols)
        miss_total = int(union_df[cols].isna().sum().sum())
        pm_rows.append({
            "platform": key,
            "n_features": len(cols),
            "n_cells_present": int(plat_pres[key].sum()),
            "n_cells_absent": n_absent,
            "frac_cells_present": float(plat_pres[key].mean()),
            "pct_missingness_from_platform_absence": (
                float(miss_from_abs / miss_total * 100) if miss_total > 0 else np.nan
            ),
        })
    platform_miss_df = pd.DataFrame(pm_rows)
    platform_miss_df.to_csv(
        OUT_REPORTS / "combined_union_platform_missingness_contrib.csv", index=False
    )
    print("\nCombined union, missingness from platform absence:")
    display(platform_miss_df)


Core cohort (RNA ∩ CNV ∩ MUT): 1079 cell lines
Group column for CV: lineage_1
Loaded prot_combined_union: (1079, 18751)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)

Selected 100 drugs for deep-imputation bake-off by PRISM AUC coverage.
MISSINGNESS REPORT
Availability matrix: artifacts/reports/notebook 5_deep_imputation/modality_availability_matrix.csv  shape=(1079, 6)

Modality availability summary:


,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72



Per-arm feature missingness:


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
1,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
2,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0



Combined union platform patterns:


,pattern,n_cells,frac_cells
0,none,400,0.370714
1,ms+rppa+procan,233,0.215941
2,rppa+procan,187,0.173309
3,rppa,128,0.118628
4,ms+rppa,64,0.059314
5,procan,60,0.055607
6,ms+procan,5,0.004634
7,ms,2,0.001854



Combined union, missingness from platform absence:


,platform,n_features,n_cells_present,n_cells_absent,frac_cells_present,pct_missingness_from_platform_absence
0,ms,10847,304,775,0.281742,92.892390
1,rppa,144,612,467,0.567192,100.000000
2,procan,7760,485,594,0.449490,78.405864


In [4]:
def build_missingness_report(
    avail_summary: pd.DataFrame,
    feat_miss_df: pd.DataFrame,
    pat_counts: Optional[pd.DataFrame],
    platform_miss_df: Optional[pd.DataFrame],
    track2_arm: str,
    all_seeds: List[int],
    ts: str,
) -> dict:
    report = {
        "generated_at": ts,
        "seeds": all_seeds,
        "active_arms": ACTIVE_ARMS,
        "deprioritised_arm": "prot_rppa_ccle",
        "leakage_note": (
            "All imputation statistics are computed inside CV folds only. "
            "No global statistics are used at any point."
        ),
        "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
        "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
    }

    if pat_counts is not None:
        report["combined_union_platform_patterns"] = {
            "arm": track2_arm,
            "structural_missingness_warning": (
                "Missingness is driven by platform availability, not random "
                "per-protein failure. Missingness indicators remain important "
                "for this arm even under deep imputation."
            ),
            "patterns": pat_counts.to_dict(orient="records"),
        }

    if platform_miss_df is not None:
        report["combined_union_platform_missingness_contrib"] = (
            platform_miss_df.to_dict(orient="records")
        )

    report["deep_bakeoff_outputs"] = {
        "detail_file": f"deep_impute_bakeoff_merged_{len(all_seeds)}seeds.csv",
        "summary_file": f"deep_impute_bakeoff_summary_merged_{len(all_seeds)}seeds.csv",
        "prot_lock_file": PROT_DEEP_LOCK_FILENAME,
        "cnv_lock_file": CNV_DEEP_LOCK_FILENAME,
    }
    return report


missingness_report = build_missingness_report(
    avail_summary=avail_summary,
    feat_miss_df=feat_miss_df,
    pat_counts=pat_counts,
    platform_miss_df=platform_miss_df,
    track2_arm=TRACK2_ARM,
    all_seeds=ALL_SEEDS,
    ts=datetime.now(timezone.utc).isoformat(),
)
report_path = OUT_REPORTS / "missingness_report.json"
write_json(missingness_report, report_path)
print(f"\nMissingness report written: {report_path}")


# Deep imputation models
class DenoisingAutoencoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, in_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ConditionalAutoencoder(nn.Module):
    def __init__(self, target_dim: int, aux_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        in_dim = target_dim + aux_dim
        self.target_dim = target_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, target_dim),
        )

    def forward(self, x_target: torch.Tensor, x_aux: torch.Tensor) -> torch.Tensor:
        x = torch.cat([x_target, x_aux], dim=1)
        return self.net(x)


class StandardiseObservedBlock:
    def __init__(self):
        self.mean_ = None
        self.std_ = None
        self.keep_ = None

    def fit(self, X: np.ndarray) -> "StandardiseObservedBlock":
        X = np.asarray(X, dtype=float)
        self.keep_ = np.isfinite(X).any(axis=0)
        if self.keep_.sum() == 0:
            self.mean_ = np.zeros(0, dtype=np.float32)
            self.std_ = np.ones(0, dtype=np.float32)
            return self
        Xk = X[:, self.keep_]
        col_mean = np.nanmean(Xk, axis=0)
        col_std = np.nanstd(Xk, axis=0)
        col_std = np.where(col_std <= 1e-8, 1.0, col_std)
        self.mean_ = col_mean.astype(np.float32)
        self.std_ = col_std.astype(np.float32)
        return self

    def transform(self, X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        X = np.asarray(X, dtype=float)
        if self.keep_ is None or self.keep_.sum() == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32), np.zeros((X.shape[0], 0), dtype=bool)
        Xk = X[:, self.keep_]
        miss = ~np.isfinite(Xk)
        Xfill = np.where(miss, self.mean_[None, :], Xk)
        Xstd = (Xfill - self.mean_[None, :]) / self.std_[None, :]
        return Xstd.astype(np.float32), miss

    def inverse_transform(self, X_std: np.ndarray) -> np.ndarray:
        if X_std.shape[1] == 0:
            return X_std.astype(np.float32)
        return (X_std * self.std_[None, :] + self.mean_[None, :]).astype(np.float32)


class DeepBlockTransformer:
    def __init__(self, n_components: int, random_state: int):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardiseObservedBlock()
        self.pca = None

    def fit(self, X_train: np.ndarray) -> "DeepBlockTransformer":
        X_train_std, _ = self.scaler.fit(X_train).transform(X_train)
        if X_train_std.shape[1] == 0:
            return self
        n, d = X_train_std.shape
        n_comp = min(self.n_components, max(1, n - 1), d)
        self.pca = CPU_PCA(n_components=n_comp, random_state=self.random_state)
        self.pca.fit(X_train_std)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        X_std, _ = self.scaler.transform(X)
        if X_std.shape[1] == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        if self.pca is None:
            return X_std.astype(np.float32)
        return self.pca.transform(X_std).astype(np.float32)



def _make_hidden_dim(in_dim: int, aux_dim: int = 0) -> int:
    raw = int((in_dim + aux_dim) * DEEP_HIDDEN_FRAC)
    return int(max(DEEP_MIN_HIDDEN, min(DEEP_MAX_HIDDEN, raw)))



def _split_train_val_idx(n_samples: int, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    idx = np.arange(n_samples)
    if n_samples < (MIN_VAL_SAMPLES * 2):
        return idx, np.array([], dtype=int)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_val = max(MIN_VAL_SAMPLES, int(round(n_samples * VAL_FRAC)))
    n_val = min(n_val, n_samples // 3)
    val_idx = np.sort(idx[:n_val])
    tr_idx = np.sort(idx[n_val:])
    return tr_idx, val_idx



def _train_autoencoder(
    X_train_std: np.ndarray,
    miss_train: np.ndarray,
    seed: int,
) -> nn.Module:
    in_dim = X_train_std.shape[1]
    hidden_dim = _make_hidden_dim(in_dim)
    model = DenoisingAutoencoder(in_dim=in_dim, hidden_dim=hidden_dim, dropout=DEEP_DROPOUT).to(TORCH_DEVICE)

    tr_idx, val_idx = _split_train_val_idx(X_train_std.shape[0], seed)
    x_tr = torch.from_numpy(X_train_std[tr_idx])
    m_tr = torch.from_numpy((~miss_train[tr_idx]).astype(np.float32))

    ds = TensorDataset(x_tr, m_tr)
    dl = DataLoader(ds, batch_size=min(DEEP_BATCH_SIZE, len(ds)), shuffle=True)

    if len(val_idx) > 0:
        x_val = torch.from_numpy(X_train_std[val_idx]).to(TORCH_DEVICE)
        m_val = torch.from_numpy((~miss_train[val_idx]).astype(np.float32)).to(TORCH_DEVICE)
    else:
        x_val, m_val = None, None

    opt = torch.optim.Adam(model.parameters(), lr=DEEP_LR, weight_decay=DEEP_WEIGHT_DECAY)
    best_state = copy.deepcopy(model.state_dict())
    best_loss = math.inf
    bad_epochs = 0

    for epoch in range(DEEP_EPOCHS):
        model.train()
        for xb, mb in dl:
            xb = xb.to(TORCH_DEVICE)
            mb = mb.to(TORCH_DEVICE)
            noisy = xb + torch.randn_like(xb) * DEEP_NOISE_STD
            pred = model(noisy)
            denom = mb.sum().clamp_min(1.0)
            loss = (((pred - xb) ** 2) * mb).sum() / denom
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            if x_val is not None:
                pred_val = model(x_val)
                denom = m_val.sum().clamp_min(1.0)
                val_loss = (((pred_val - x_val) ** 2) * m_val).sum() / denom
                current = float(val_loss.item())
            else:
                pred_tr = model(torch.from_numpy(X_train_std).to(TORCH_DEVICE))
                m_all = torch.from_numpy((~miss_train).astype(np.float32)).to(TORCH_DEVICE)
                denom = m_all.sum().clamp_min(1.0)
                tr_loss = (((pred_tr - torch.from_numpy(X_train_std).to(TORCH_DEVICE)) ** 2) * m_all).sum() / denom
                current = float(tr_loss.item())

        if current < (best_loss - 1e-6):
            best_loss = current
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= DEEP_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model



def _train_conditional_autoencoder(
    X_target_train_std: np.ndarray,
    miss_train: np.ndarray,
    X_aux_train: np.ndarray,
    seed: int,
) -> nn.Module:
    target_dim = X_target_train_std.shape[1]
    aux_dim = X_aux_train.shape[1]
    hidden_dim = _make_hidden_dim(target_dim, aux_dim)
    model = ConditionalAutoencoder(
        target_dim=target_dim,
        aux_dim=aux_dim,
        hidden_dim=hidden_dim,
        dropout=DEEP_DROPOUT,
    ).to(TORCH_DEVICE)

    tr_idx, val_idx = _split_train_val_idx(X_target_train_std.shape[0], seed)
    xt_tr = torch.from_numpy(X_target_train_std[tr_idx])
    xa_tr = torch.from_numpy(X_aux_train[tr_idx].astype(np.float32))
    m_tr = torch.from_numpy((~miss_train[tr_idx]).astype(np.float32))

    ds = TensorDataset(xt_tr, xa_tr, m_tr)
    dl = DataLoader(ds, batch_size=min(DEEP_BATCH_SIZE, len(ds)), shuffle=True)

    if len(val_idx) > 0:
        xt_val = torch.from_numpy(X_target_train_std[val_idx]).to(TORCH_DEVICE)
        xa_val = torch.from_numpy(X_aux_train[val_idx].astype(np.float32)).to(TORCH_DEVICE)
        m_val = torch.from_numpy((~miss_train[val_idx]).astype(np.float32)).to(TORCH_DEVICE)
    else:
        xt_val, xa_val, m_val = None, None, None

    opt = torch.optim.Adam(model.parameters(), lr=DEEP_LR, weight_decay=DEEP_WEIGHT_DECAY)
    best_state = copy.deepcopy(model.state_dict())
    best_loss = math.inf
    bad_epochs = 0

    for epoch in range(DEEP_EPOCHS):
        model.train()
        for xtb, xab, mb in dl:
            xtb = xtb.to(TORCH_DEVICE)
            xab = xab.to(TORCH_DEVICE)
            mb = mb.to(TORCH_DEVICE)
            noisy = xtb + torch.randn_like(xtb) * DEEP_NOISE_STD
            pred = model(noisy, xab)
            denom = mb.sum().clamp_min(1.0)
            loss = (((pred - xtb) ** 2) * mb).sum() / denom
            opt.zero_grad()
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            if xt_val is not None:
                pred_val = model(xt_val, xa_val)
                denom = m_val.sum().clamp_min(1.0)
                val_loss = (((pred_val - xt_val) ** 2) * m_val).sum() / denom
                current = float(val_loss.item())
            else:
                xt_all = torch.from_numpy(X_target_train_std).to(TORCH_DEVICE)
                xa_all = torch.from_numpy(X_aux_train.astype(np.float32)).to(TORCH_DEVICE)
                m_all = torch.from_numpy((~miss_train).astype(np.float32)).to(TORCH_DEVICE)
                pred_all = model(xt_all, xa_all)
                denom = m_all.sum().clamp_min(1.0)
                tr_loss = (((pred_all - xt_all) ** 2) * m_all).sum() / denom
                current = float(tr_loss.item())

        if current < (best_loss - 1e-6):
            best_loss = current
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= DEEP_PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model



def deep_impute_block(
    X_train: np.ndarray,
    X_all: np.ndarray,
    random_state: int,
    add_indicators: bool,
    force_indicators: bool,
    n_pca_components: int,
) -> np.ndarray:
    scaler = StandardiseObservedBlock().fit(X_train)
    Xtr_std, miss_tr = scaler.transform(X_train)
    Xal_std, miss_al = scaler.transform(X_all)

    if Xtr_std.shape[1] == 0:
        return np.zeros((X_all.shape[0], 0), dtype=np.float32)

    model = _train_autoencoder(X_train_std=Xtr_std, miss_train=miss_tr, seed=random_state)
    with torch.no_grad():
        Xal_pred = model(torch.from_numpy(Xal_std).to(TORCH_DEVICE)).cpu().numpy().astype(np.float32)

    Xal_imp_std = Xal_std.copy()
    Xal_imp_std[miss_al] = Xal_pred[miss_al]

    if add_indicators or force_indicators:
        ind = miss_al.astype(np.float32)
        ind_any = ind.any(axis=0)
        ind = ind[:, ind_any]
        if ind.shape[1] > 0:
            ind_sc = StandardScaler()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ind = ind_sc.fit_transform(ind).astype(np.float32)
                Xal_imp_std = np.concatenate([Xal_imp_std, ind], axis=1)

    Xtr_imp_std = Xtr_std.copy()
    with torch.no_grad():
        Xtr_pred = model(torch.from_numpy(Xtr_std).to(TORCH_DEVICE)).cpu().numpy().astype(np.float32)
    Xtr_imp_std[miss_tr] = Xtr_pred[miss_tr]

    if add_indicators or force_indicators:
        ind_tr = miss_tr.astype(np.float32)
        ind_any_tr = ind_tr.any(axis=0)
        ind_tr = ind_tr[:, ind_any_tr]
        if ind_tr.shape[1] > 0:
            ind_sc = StandardScaler()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ind_sc.fit(ind_tr)
                Xtr_imp_std = np.concatenate([Xtr_imp_std, ind_sc.transform(ind_tr).astype(np.float32)], axis=1)
                ind_al = miss_al.astype(np.float32)[:, ind_any_tr]
                Xal_imp_std = np.concatenate([Xal_std.copy(), ind_sc.transform(ind_al).astype(np.float32)], axis=1)
                Xal_imp_std[:, :Xtr_std.shape[1]][miss_al] = Xal_pred[miss_al]

    n, d = Xtr_imp_std.shape
    n_comp = min(n_pca_components, max(1, n - 1), d)
    pca = CPU_PCA(n_components=n_comp, random_state=random_state)
    pca.fit(Xtr_imp_std)
    return pca.transform(Xal_imp_std).astype(np.float32)



def deep_impute_block_with_aux(
    X_target_train: np.ndarray,
    X_target_all: np.ndarray,
    X_aux_train: np.ndarray,
    X_aux_all: np.ndarray,
    random_state: int,
    add_indicators: bool,
    force_indicators: bool,
    n_pca_components: int,
) -> np.ndarray:
    scaler = StandardiseObservedBlock().fit(X_target_train)
    Xt_tr_std, miss_tr = scaler.transform(X_target_train)
    Xt_al_std, miss_al = scaler.transform(X_target_all)

    if Xt_tr_std.shape[1] == 0:
        return np.zeros((X_target_all.shape[0], 0), dtype=np.float32)

    model = _train_conditional_autoencoder(
        X_target_train_std=Xt_tr_std,
        miss_train=miss_tr,
        X_aux_train=X_aux_train,
        seed=random_state,
    )

    with torch.no_grad():
        Xt_al_pred = model(
            torch.from_numpy(Xt_al_std).to(TORCH_DEVICE),
            torch.from_numpy(X_aux_all.astype(np.float32)).to(TORCH_DEVICE),
        ).cpu().numpy().astype(np.float32)
        Xt_tr_pred = model(
            torch.from_numpy(Xt_tr_std).to(TORCH_DEVICE),
            torch.from_numpy(X_aux_train.astype(np.float32)).to(TORCH_DEVICE),
        ).cpu().numpy().astype(np.float32)

    Xt_tr_imp_std = Xt_tr_std.copy()
    Xt_al_imp_std = Xt_al_std.copy()
    Xt_tr_imp_std[miss_tr] = Xt_tr_pred[miss_tr]
    Xt_al_imp_std[miss_al] = Xt_al_pred[miss_al]

    if add_indicators or force_indicators:
        ind_tr = miss_tr.astype(np.float32)
        ind_any = ind_tr.any(axis=0)
        ind_tr = ind_tr[:, ind_any]
        ind_al = miss_al.astype(np.float32)[:, ind_any]
        if ind_tr.shape[1] > 0:
            ind_sc = StandardScaler()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                Xt_tr_imp_std = np.concatenate([Xt_tr_imp_std, ind_sc.fit_transform(ind_tr).astype(np.float32)], axis=1)
                Xt_al_imp_std = np.concatenate([Xt_al_imp_std, ind_sc.transform(ind_al).astype(np.float32)], axis=1)

    n, d = Xt_tr_imp_std.shape
    n_comp = min(n_pca_components, max(1, n - 1), d)
    pca = CPU_PCA(n_components=n_comp, random_state=random_state)
    pca.fit(Xt_tr_imp_std)
    return pca.transform(Xt_al_imp_std).astype(np.float32)



def passthrough_transform_block(
    X_train: np.ndarray,
    X_all: np.ndarray,
    random_state: int,
    n_pca_components: int,
) -> np.ndarray:
    tr = DeepBlockTransformer(n_components=n_pca_components, random_state=random_state).fit(X_train)
    return tr.transform(X_all)


DEEP_PROT_STRATEGIES = [
    ("deep_dae", False),
    ("deep_dae+indicators", True),
]

DEEP_CNV_STRATEGIES = [
    ("deep_cnv_dae", False),
    ("deep_cnv_dae+indicators", True),
]

USE_AUX_FOR_PROT_DEEP_IMPUTATION = True
AUX_FOR_PROT_DEEP_NAME = "deep_aux_rna_cnv_mut"


# Cache helpers

def _prot_deep_cache_path(seed: int, arm: str, fold_i: int, strat_name: str, force_indicators: bool) -> Path:
    return OUT_CACHE / (
        f"prot_deep__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}"
        f"__strat{_safe_name(strat_name)}__forceind{int(force_indicators)}.joblib"
    )



def _prot_deep_aux_cache_path(seed: int, arm: str, fold_i: int, strat_name: str, force_indicators: bool, aux_name: str) -> Path:
    return OUT_CACHE / (
        f"prot_deep_aux__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}"
        f"__strat{_safe_name(strat_name)}__forceind{int(force_indicators)}__aux{_safe_name(aux_name)}.joblib"
    )



def _cnv_deep_cache_path(seed: int, arm: str, fold_i: int, strat_name: str, add_indicators: bool) -> Path:
    return OUT_CACHE / (
        f"cnv_deep__{CACHE_TAG}__seed{seed}__arm{_safe_name(arm)}__fold{fold_i}"
        f"__strat{_safe_name(strat_name)}__addind{int(add_indicators)}.joblib"
    )



def load_or_build_prot_deep_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_deep_cache_path(seed, arm, fold_i, strat_name, force_indicators)
    if path.exists():
        return load(path)

    Xp = deep_impute_block(
        X_train=prot_elig_values[idx_train_full],
        X_all=prot_elig_values,
        random_state=seed + 300,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
    )
    dump(Xp, path)
    return Xp



def load_or_build_prot_deep_aux_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    aux_name: str,
    add_indicators: bool,
    force_indicators: bool,
    prot_elig_values: np.ndarray,
    aux_all_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _prot_deep_aux_cache_path(seed, arm, fold_i, strat_name, force_indicators, aux_name)
    if path.exists():
        return load(path)

    Xp = deep_impute_block_with_aux(
        X_target_train=prot_elig_values[idx_train_full],
        X_target_all=prot_elig_values,
        X_aux_train=aux_all_values[idx_train_full],
        X_aux_all=aux_all_values,
        random_state=seed + 330,
        add_indicators=add_indicators,
        force_indicators=force_indicators,
        n_pca_components=PCA_COMPONENTS,
    )
    dump(Xp, path)
    return Xp



def load_or_build_cnv_deep_fold_cache(
    seed: int,
    arm: str,
    fold_i: int,
    strat_name: str,
    add_indicators: bool,
    cnv_elig_values: np.ndarray,
    idx_train_full: np.ndarray,
) -> np.ndarray:
    path = _cnv_deep_cache_path(seed, arm, fold_i, strat_name, add_indicators)
    if path.exists():
        return load(path)

    Xc = deep_impute_block(
        X_train=cnv_elig_values[idx_train_full],
        X_all=cnv_elig_values,
        random_state=seed + 360,
        add_indicators=add_indicators,
        force_indicators=False,
        n_pca_components=PCA_COMPONENTS,
    )
    dump(Xc, path)
    return Xc






Missingness report written: artifacts/reports/notebook 5_deep_imputation/missingness_report.json


In [5]:
# Deep proteomics imputation bake-off
print("DEEP IMPUTATION BAKE-OFF FOR PROTEOMICS (3 seeds, leakage-safe)")

all_bakeoff_rows: List[dict] = []
seeds_to_run: List[int] = []
REQUIRED_NEW_COLS = {"config_rank", "model", "feature_set", "uses_prot"}

for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"deep_impute_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)
        if not REQUIRED_NEW_COLS.issubset(existing.columns):
            print(f"[resume] Seed {run_seed} file exists but is from old schema, rerunning.")
            seeds_to_run.append(run_seed)
            continue
        existing["seed"] = existing["seed"].astype(int)
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping deep proteomics bake-off loop.")
else:
    print(f"[resume] Seeds remaining: {seeds_to_run}")

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded.")
            continue

        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            print(f"  [SKIP] {arm} has no configs assigned.")
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_ckpt_path = OUT_REPORTS / f"deep_impute_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs: set = set()

        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)
            if REQUIRED_NEW_COLS.issubset(arm_existing.columns):
                arm_existing["seed"] = arm_existing["seed"].astype(int)
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records"))
                    already_done_drugs = set(arm_existing["compound_id"].unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed)
        print(f"  {arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig = prot_core.loc[eligible_cells]
        prot_elig_values = prot_elig.to_numpy(dtype=float)
        fold_cache: Dict[int, dict] = {}
        force_indicators = (arm == "prot_combined_union")

        print(f"    Building/loading deep fold caches for {arm}...")
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]

            base_payload = load_or_build_base_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
            )

            X_aux_all = np.concatenate([base_payload["Xr"], base_payload["Xc"], base_payload["Xm"]], axis=1)

            fold_entry = {
                "base_mats": {
                    "rna": base_payload["Xr"],
                    "cnv": base_payload["Xc"],
                    "mut": base_payload["Xm"],
                },
                "prot": {},
                "prot_aux": {},
            }

            for strat_name, add_ind in DEEP_PROT_STRATEGIES:
                try:
                    Xp = load_or_build_prot_deep_fold_cache(
                        seed=run_seed,
                        arm=arm,
                        fold_i=fold_i,
                        strat_name=strat_name,
                        add_indicators=add_ind,
                        force_indicators=force_indicators,
                        prot_elig_values=prot_elig_values,
                        idx_train_full=np.asarray(train_idx, dtype=int),
                    )
                except Exception as e:
                    print(f"    [WARN] deep cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                    Xp = None

                fold_entry["prot"][(strat_name, add_ind, force_indicators)] = Xp

                if USE_AUX_FOR_PROT_DEEP_IMPUTATION:
                    try:
                        Xp_aux = load_or_build_prot_deep_aux_fold_cache(
                            seed=run_seed,
                            arm=arm,
                            fold_i=fold_i,
                            strat_name=strat_name,
                            aux_name=AUX_FOR_PROT_DEEP_NAME,
                            add_indicators=add_ind,
                            force_indicators=force_indicators,
                            prot_elig_values=prot_elig_values,
                            aux_all_values=X_aux_all,
                            idx_train_full=np.asarray(train_idx, dtype=int),
                        )
                    except Exception as e:
                        print(f"    [WARN] deep aux cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                        Xp_aux = None

                    fold_entry["prot_aux"][(strat_name, add_ind, force_indicators)] = Xp_aux

            fold_cache[fold_i] = fold_entry

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            if drug in already_done_drugs:
                continue

            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1

            if n_run % 5 == 0:
                arm_rows_partial = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
                if arm_rows_partial:
                    pd.DataFrame(arm_rows_partial).to_csv(arm_ckpt_path, index=False)
                    print(f"    [mid-checkpoint] drug {n_run}/{N_DRUGS_BAKEOFF} | {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]
                base_mats = fold_cache[fold_i]["base_mats"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    uses_prot = "prot" in cfg_keys

                    X_nonprot = concat_selected_modalities(
                        mats=base_mats,
                        selected_keys=tuple(k for k in cfg_keys if k != "prot"),
                        n_rows=len(eligible_cells),
                    )

                    if not uses_prot:
                        if X_nonprot.shape[1] == 0:
                            continue
                        mdl = make_model(cfg_model, SEED)
                        mdl.fit(X_nonprot[idx_train], y_train)
                        pred = mdl.predict(X_nonprot[idx_test])
                        all_bakeoff_rows.append({
                            "seed": run_seed,
                            "config_rank": cfg_rank,
                            "arm": arm,
                            "model": cfg_model,
                            "feature_set": cfg_feature_set,
                            "uses_prot": False,
                            "compound_id": drug,
                            "fold": fold_i,
                            "imputer_strategy": "no_prot_reference",
                            "add_indicators": False,
                            "force_indicators": False,
                            "n_train": n_train,
                            "n_test": n_test,
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })
                        continue

                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    if len(cfg_keys_wo_prot) > 0:
                        X_ref = concat_selected_modalities(base_mats, cfg_keys_wo_prot, len(eligible_cells))
                        if X_ref.shape[1] > 0:
                            mdl_ref = make_model(cfg_model, SEED)
                            mdl_ref.fit(X_ref[idx_train], y_train)
                            pred_ref = mdl_ref.predict(X_ref[idx_test])
                            all_bakeoff_rows.append({
                                "seed": run_seed,
                                "config_rank": cfg_rank,
                                "arm": arm,
                                "model": cfg_model,
                                "feature_set": cfg_feature_set,
                                "uses_prot": True,
                                "compound_id": drug,
                                "fold": fold_i,
                                "imputer_strategy": "reference_without_prot",
                                "add_indicators": False,
                                "force_indicators": False,
                                "n_train": n_train,
                                "n_test": n_test,
                                "spearman": spearman_corr(y_test, pred_ref),
                                "r2": float(r2_score(y_test, pred_ref)),
                            })

                    for strat_name, add_ind in DEEP_PROT_STRATEGIES:
                        Xp = fold_cache[fold_i]["prot"].get((strat_name, add_ind, force_indicators))
                        if Xp is not None and Xp.shape[1] > 0:
                            parts = []
                            bad_cfg = False
                            for k in cfg_keys:
                                if k == "prot":
                                    parts.append(Xp)
                                else:
                                    arr = base_mats.get(k)
                                    if arr is None or arr.shape[1] == 0:
                                        bad_cfg = True
                                        break
                                    parts.append(arr)
                            if not bad_cfg and len(parts) > 0:
                                Xf = np.concatenate(parts, axis=1)
                                mdl = make_model(cfg_model, SEED)
                                mdl.fit(Xf[idx_train], y_train)
                                pred = mdl.predict(Xf[idx_test])
                                all_bakeoff_rows.append({
                                    "seed": run_seed,
                                    "config_rank": cfg_rank,
                                    "arm": arm,
                                    "model": cfg_model,
                                    "feature_set": cfg_feature_set,
                                    "uses_prot": True,
                                    "compound_id": drug,
                                    "fold": fold_i,
                                    "imputer_strategy": strat_name,
                                    "add_indicators": add_ind,
                                    "force_indicators": force_indicators,
                                    "n_train": n_train,
                                    "n_test": n_test,
                                    "spearman": spearman_corr(y_test, pred),
                                    "r2": float(r2_score(y_test, pred)),
                                })

                        if USE_AUX_FOR_PROT_DEEP_IMPUTATION:
                            Xp_aux = fold_cache[fold_i]["prot_aux"].get((strat_name, add_ind, force_indicators))
                            if Xp_aux is not None and Xp_aux.shape[1] > 0:
                                parts_aux = []
                                bad_cfg_aux = False
                                for k in cfg_keys:
                                    if k == "prot":
                                        parts_aux.append(Xp_aux)
                                    else:
                                        arr = base_mats.get(k)
                                        if arr is None or arr.shape[1] == 0:
                                            bad_cfg_aux = True
                                            break
                                        parts_aux.append(arr)
                                if not bad_cfg_aux and len(parts_aux) > 0:
                                    Xf_aux = np.concatenate(parts_aux, axis=1)
                                    mdl_aux = make_model(cfg_model, SEED)
                                    mdl_aux.fit(Xf_aux[idx_train], y_train)
                                    pred_aux = mdl_aux.predict(Xf_aux[idx_test])
                                    all_bakeoff_rows.append({
                                        "seed": run_seed,
                                        "config_rank": cfg_rank,
                                        "arm": arm,
                                        "model": cfg_model,
                                        "feature_set": cfg_feature_set,
                                        "uses_prot": True,
                                        "compound_id": drug,
                                        "fold": fold_i,
                                        "imputer_strategy": f"{AUX_FOR_PROT_DEEP_NAME}::{strat_name}",
                                        "add_indicators": add_ind,
                                        "force_indicators": force_indicators,
                                        "n_train": n_train,
                                        "n_test": n_test,
                                        "spearman": spearman_corr(y_test, pred_aux),
                                        "r2": float(r2_score(y_test, pred_aux)),
                                    })

        print(f"    drugs_run={n_run}, drugs_skipped={n_skip}")
        arm_rows = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
        arm_df = pd.DataFrame(arm_rows)
        arm_df.to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}  shape={arm_df.shape}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"deep_impute_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")

bakeoff_df = pd.DataFrame(all_bakeoff_rows)
merged_path = OUT_REPORTS / f"deep_impute_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print(f"\nMerged deep proteomics bake-off: {merged_path}  shape={bakeoff_df.shape}")

drug_means = (
    bakeoff_df
    .groupby(["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy", "compound_id"], as_index=False)
    .agg(
        spearman=("spearman", lambda x: float(np.nanmean(x))),
        r2=("r2", lambda x: float(np.nanmean(x))),
        n_folds=("fold", "nunique"),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(["config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", lambda x: float(np.nanmean(x))),
        median_spearman=("spearman", lambda x: float(np.nanmedian(x))),
        std_spearman=("spearman", lambda x: float(np.nanstd(x))),
        mean_r2=("r2", lambda x: float(np.nanmean(x))),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

base_ref = (
    bakeoff_summary[
        bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])
    ][["config_rank", "arm", "model", "feature_set", "mean_spearman", "imputer_strategy"]]
    .rename(columns={"mean_spearman": "baseline_mean_spearman", "imputer_strategy": "baseline_strategy"})
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

bakeoff_summary = bakeoff_summary.merge(
    base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)

bakeoff_summary["delta_vs_baseline"] = np.where(
    bakeoff_summary["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"]),
    0.0,
    bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"],
)

summary_path = OUT_REPORTS / f"deep_impute_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
print("\nDeep proteomics bake-off summary:")
display(bakeoff_summary)

per_seed_summary = (
    drug_means
    .groupby(["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "imputer_strategy"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", lambda x: float(np.nanmean(x))),
        median_spearman=("spearman", lambda x: float(np.nanmedian(x))),
        std_spearman=("spearman", lambda x: float(np.nanstd(x))),
    )
    .sort_values(["config_rank", "seed", "mean_spearman"], ascending=[True, True, False])
)

per_seed_path = OUT_REPORTS / "deep_impute_bakeoff_per_seed_summary.csv"
per_seed_summary.to_csv(per_seed_path, index=False)
print("\nPer-seed deep proteomics summary:")
display(per_seed_summary)


# Deep CNV imputation sensitivity
print("DEEP CNV IMPUTATION SENSITIVITY BAKE-OFF (3 seeds, leakage-safe)")
cnv_bakeoff_rows: List[dict] = []

for run_seed in ALL_SEEDS:
    set_seeds(run_seed)
    print(f"  Seed {run_seed}")

    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            continue

        arm_cfgs = CNV_CONFIGS_BY_ARM.get(arm, [])
        if not arm_cfgs:
            continue

        prot_df = prot_arm_data[arm].copy()
        prot_df.index = prot_df.index.astype(str).str.strip()
        prot_core = prot_df.reindex(core_cells)

        has_prot = prot_core.notna().any(axis=1)
        eligible_cells = sorted(has_prot[has_prot].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] CNV/{arm}: only {len(eligible_cells)} eligible cells.")
            continue

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed)
        print(f"  CNV/{arm}: {split_name}, {len(eligible_cells)} cells")

        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig_df = prot_core.loc[eligible_cells]
        cnv_elig_values = cnv.loc[eligible_cells].to_numpy(dtype=float)

        cnv_fold_cache: Dict[int, dict] = {}
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]

            base_payload = load_or_build_base_fold_cache(
                seed=run_seed,
                arm=arm,
                fold_i=fold_i,
                train_cells=train_cells,
                eligible_cells=eligible_cells,
            )

            Xp_base = passthrough_transform_block(
                X_train=prot_elig_df.loc[train_cells].to_numpy(dtype=float),
                X_all=prot_elig_df.loc[eligible_cells].to_numpy(dtype=float),
                random_state=run_seed + 500,
                n_pca_components=PCA_COMPONENTS,
            )

            fold_entry = {
                "base_mats": {
                    "rna": base_payload["Xr"],
                    "mut": base_payload["Xm"],
                    "prot": Xp_base,
                },
                "cnv": {},
            }

            for strat_name, add_ind in DEEP_CNV_STRATEGIES:
                try:
                    Xc_imp = load_or_build_cnv_deep_fold_cache(
                        seed=run_seed,
                        arm=arm,
                        fold_i=fold_i,
                        strat_name=strat_name,
                        add_indicators=add_ind,
                        cnv_elig_values=cnv_elig_values,
                        idx_train_full=np.asarray(train_idx, dtype=int),
                    )
                except Exception as e:
                    print(f"    [WARN] deep CNV cache build failed {arm}/{strat_name}/fold{fold_i}: {e}")
                    Xc_imp = None
                fold_entry["cnv"][(strat_name, add_ind)] = Xc_imp

            cnv_fold_cache[fold_i] = fold_entry

        n_run = 0
        n_skip = 0

        for drug in selected_drugs:
            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1
                continue

            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist()
            y_all = df["y"].to_numpy(dtype=float)

            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]
            y_all = y_all[valid]
            fold_ids = fold_ids[valid]

            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1
                continue

            n_run += 1
            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test = int(in_test.sum())
                n_train = int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue

                idx_train = idx_all[~in_test]
                idx_test = idx_all[in_test]
                y_train = y_all[~in_test]
                y_test = y_all[in_test]

                base_mats = cnv_fold_cache[fold_i]["base_mats"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)

                    cfg_keys_wo_cnv = tuple(k for k in cfg_keys if k != "cnv")
                    ref_parts = []
                    bad_ref = False
                    for k in cfg_keys_wo_cnv:
                        arr = base_mats.get(k)
                        if arr is None or arr.shape[1] == 0:
                            bad_ref = True
                            break
                        ref_parts.append(arr)

                    if not bad_ref and len(ref_parts) > 0:
                        X_ref = np.concatenate(ref_parts, axis=1)
                        mdl_ref = make_model(cfg_model, SEED)
                        mdl_ref.fit(X_ref[idx_train], y_train)
                        pred_ref = mdl_ref.predict(X_ref[idx_test])
                        cnv_bakeoff_rows.append({
                            "seed": run_seed,
                            "config_rank": cfg_rank,
                            "arm": arm,
                            "model": cfg_model,
                            "feature_set": cfg_feature_set,
                            "compound_id": drug,
                            "fold": fold_i,
                            "imputer_strategy": "reference_without_cnv",
                            "n_train": n_train,
                            "n_test": n_test,
                            "spearman": spearman_corr(y_test, pred_ref),
                            "r2": float(r2_score(y_test, pred_ref)),
                        })

                    for strat_name, add_ind in DEEP_CNV_STRATEGIES:
                        Xc_imp = cnv_fold_cache[fold_i]["cnv"].get((strat_name, add_ind))
                        if Xc_imp is None or Xc_imp.shape[1] == 0:
                            continue

                        parts = []
                        bad_cfg = False
                        for k in cfg_keys:
                            if k == "cnv":
                                parts.append(Xc_imp)
                            else:
                                arr = base_mats.get(k)
                                if arr is None or arr.shape[1] == 0:
                                    bad_cfg = True
                                    break
                                parts.append(arr)

                        if bad_cfg or len(parts) == 0:
                            continue

                        Xf = np.concatenate(parts, axis=1)
                        mdl = make_model(cfg_model, SEED)
                        mdl.fit(Xf[idx_train], y_train)
                        pred = mdl.predict(Xf[idx_test])
                        cnv_bakeoff_rows.append({
                            "seed": run_seed,
                            "config_rank": cfg_rank,
                            "arm": arm,
                            "model": cfg_model,
                            "feature_set": cfg_feature_set,
                            "compound_id": drug,
                            "fold": fold_i,
                            "imputer_strategy": f"cnv::{strat_name}",
                            "n_train": n_train,
                            "n_test": n_test,
                            "spearman": spearman_corr(y_test, pred),
                            "r2": float(r2_score(y_test, pred)),
                        })

        print(f"    CNV/{arm}: drugs_run={n_run}, drugs_skipped={n_skip}")

cnv_bakeoff_df = pd.DataFrame(cnv_bakeoff_rows)
cnv_merged_path = OUT_REPORTS / f"deep_cnv_impute_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
cnv_bakeoff_df.to_csv(cnv_merged_path, index=False)
print(f"\nDeep CNV bake-off merged: {cnv_merged_path}  shape={cnv_bakeoff_df.shape}")

cnv_drug_means = (
    cnv_bakeoff_df
    .groupby(["seed", "config_rank", "arm", "model", "feature_set", "imputer_strategy", "compound_id"], as_index=False)
    .agg(
        spearman=("spearman", lambda x: float(np.nanmean(x))),
        r2=("r2", lambda x: float(np.nanmean(x))),
        n_folds=("fold", "nunique"),
    )
)

cnv_summary = (
    cnv_drug_means
    .groupby(["config_rank", "arm", "model", "feature_set", "imputer_strategy"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", lambda x: float(np.nanmean(x))),
        median_spearman=("spearman", lambda x: float(np.nanmedian(x))),
        std_spearman=("spearman", lambda x: float(np.nanstd(x))),
        mean_r2=("r2", lambda x: float(np.nanmean(x))),
    )
    .sort_values(["config_rank", "mean_spearman"], ascending=[True, False])
)

cnv_base_ref = (
    cnv_summary[cnv_summary["imputer_strategy"] == "reference_without_cnv"][["config_rank", "arm", "model", "feature_set", "mean_spearman"]]
    .rename(columns={"mean_spearman": "baseline_mean_spearman"})
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set"])
)

cnv_summary = cnv_summary.merge(
    cnv_base_ref,
    on=["config_rank", "arm", "model", "feature_set"],
    how="left",
)

cnv_summary["delta_vs_baseline"] = np.where(
    cnv_summary["imputer_strategy"] == "reference_without_cnv",
    0.0,
    cnv_summary["mean_spearman"] - cnv_summary["baseline_mean_spearman"],
)

cnv_summary_path = OUT_REPORTS / f"deep_cnv_impute_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
cnv_summary.to_csv(cnv_summary_path, index=False)
print("\nDeep CNV bake-off summary:")
display(cnv_summary)


# Lock decisions
print("DEEP IMPUTER LOCK DECISION")

prot_deep_choice = {}
for cfg in TOP10_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    cfg_uses_prot = "prot" in parse_feature_set(cfg_feature_set)
    key = f"rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = bakeoff_summary[
        (bakeoff_summary["config_rank"] == cfg_rank) &
        (bakeoff_summary["arm"] == cfg_arm) &
        (bakeoff_summary["model"] == cfg_model) &
        (bakeoff_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        prot_deep_choice[key] = {"chosen_strategy": None, "reason": "No deep bake-off rows produced for this config."}
        continue

    if not cfg_uses_prot:
        ref_rows = cfg_df[cfg_df["imputer_strategy"] == "no_prot_reference"]
        ref_sp = float(ref_rows["mean_spearman"].iloc[0]) if len(ref_rows) else np.nan
        prot_deep_choice[key] = {
            "config_rank": cfg_rank,
            "arm": cfg_arm,
            "model": cfg_model,
            "feature_set": cfg_feature_set,
            "chosen_strategy": "not_applicable",
            "mean_spearman_chosen": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "mean_spearman_baseline": round(ref_sp, 6) if np.isfinite(ref_sp) else None,
            "delta_vs_baseline": 0.0,
            "n_seeds_in_estimate": len(ALL_SEEDS),
            "reason": "This ranked configuration does not contain proteomics, so deep proteomics imputation is not applicable.",
        }
        print(f"\n{key}:")
        print("  Chosen   : not_applicable")
        continue

    candidates = cfg_df[~cfg_df["imputer_strategy"].isin(["no_prot_reference", "reference_without_prot"])].copy()
    if cfg_arm == "prot_combined_union":
        ind_candidates = candidates[candidates["imputer_strategy"].str.contains("indicator", regex=False)]
        if ind_candidates.shape[0] > 0:
            candidates = ind_candidates

    if candidates.shape[0] == 0:
        prot_deep_choice[key] = {"chosen_strategy": None, "reason": "No deep imputation candidates available for this config."}
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_prot"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan
    chosen = str(best["imputer_strategy"])
    mean_sp = float(best["mean_spearman"])
    std_sp = float(best["std_spearman"])
    delta = float(best["delta_vs_baseline"]) if pd.notna(best["delta_vs_baseline"]) else np.nan

    prot_deep_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": chosen,
        "mean_spearman_chosen": round(mean_sp, 6),
        "std_spearman_chosen": round(std_sp, 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(delta, 6) if np.isfinite(delta) else None,
        "n_seeds_in_estimate": len(ALL_SEEDS),
        "reason": (
            f"Best mean Spearman ({mean_sp:.4f} ± {std_sp:.4f}) across {len(ALL_SEEDS)} seeds"
            + (
                f"; delta vs config-specific no-prot reference: {delta:+.4f}."
                if np.isfinite(delta) else
                "; no config-specific no-prot reference available."
            )
        ),
    }
    print(f"\n{key}:")
    print(f"  Chosen   : {chosen}")
    print(f"  Spearman : {mean_sp:.4f} ± {std_sp:.4f}")
    if np.isfinite(base_sp):
        print(f"  Baseline : {base_sp:.4f}  delta={delta:+.4f}")

prot_deep_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "pca_components": PCA_COMPONENTS,
    "deep_settings": {
        "epochs": DEEP_EPOCHS,
        "patience": DEEP_PATIENCE,
        "batch_size": DEEP_BATCH_SIZE,
        "lr": DEEP_LR,
        "weight_decay": DEEP_WEIGHT_DECAY,
        "noise_std": DEEP_NOISE_STD,
        "dropout": DEEP_DROPOUT,
    },
    "top10_configs": TOP10_CONFIGS,
    "note": (
        "Deep denoising autoencoders are fitted only on training folds. "
        "For proteomics-containing configs, deep imputation strategies are compared within the exact ranked config. "
        "Auxiliary deep imputation uses RNA, CNV, and mutation embeddings from the same training fold only."
    ),
    "config_choices": prot_deep_choice,
}

prot_deep_choice_path = OUT_META / PROT_DEEP_LOCK_FILENAME
write_json(prot_deep_choice_doc, prot_deep_choice_path)
print(f"\nDeep proteomics imputer choice written: {prot_deep_choice_path}")

cnv_deep_choice = {}
for cfg in CNV_TEST_CONFIGS:
    cfg_rank = int(cfg["rank"])
    cfg_arm = str(cfg["arm"])
    cfg_model = str(cfg["model"]).lower()
    cfg_feature_set = str(cfg["feature_set"])
    key = f"cnv_rank{cfg_rank}__{cfg_arm}__{cfg_model}__{cfg_feature_set}"

    cfg_df = cnv_summary[
        (cnv_summary["config_rank"] == cfg_rank) &
        (cnv_summary["arm"] == cfg_arm) &
        (cnv_summary["model"] == cfg_model) &
        (cnv_summary["feature_set"] == cfg_feature_set)
    ].copy()

    if cfg_df.shape[0] == 0:
        cnv_deep_choice[key] = {"chosen_strategy": None, "reason": "No deep CNV bake-off rows produced for this config."}
        continue

    candidates = cfg_df[cfg_df["imputer_strategy"] != "reference_without_cnv"].copy()
    if candidates.shape[0] == 0:
        cnv_deep_choice[key] = {"chosen_strategy": None, "reason": "No deep CNV imputation candidates available for this config."}
        continue

    best = candidates.sort_values("mean_spearman", ascending=False).iloc[0]
    base_rows = cfg_df[cfg_df["imputer_strategy"] == "reference_without_cnv"]
    base_sp = float(base_rows["mean_spearman"].iloc[0]) if len(base_rows) else np.nan

    cnv_deep_choice[key] = {
        "config_rank": cfg_rank,
        "arm": cfg_arm,
        "model": cfg_model,
        "feature_set": cfg_feature_set,
        "chosen_strategy": str(best["imputer_strategy"]),
        "mean_spearman_chosen": round(float(best["mean_spearman"]), 6),
        "std_spearman_chosen": round(float(best["std_spearman"]), 6),
        "mean_spearman_baseline": round(base_sp, 6) if np.isfinite(base_sp) else None,
        "delta_vs_baseline": round(float(best["delta_vs_baseline"]), 6) if pd.notna(best["delta_vs_baseline"]) else None,
    }

cnv_deep_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "primary_target": PRIMARY_TARGET,
    "configs": CNV_TEST_CONFIGS,
    "deep_settings": {
        "epochs": DEEP_EPOCHS,
        "patience": DEEP_PATIENCE,
        "batch_size": DEEP_BATCH_SIZE,
        "lr": DEEP_LR,
        "weight_decay": DEEP_WEIGHT_DECAY,
        "noise_std": DEEP_NOISE_STD,
        "dropout": DEEP_DROPOUT,
    },
    "choices": cnv_deep_choice,
    "note": (
        "This is a separate deep CNV-imputation sensitivity analysis. "
        "It does not replace the proteomics deep-imputation bake-off."
    ),
}

cnv_deep_choice_path = OUT_META / CNV_DEEP_LOCK_FILENAME
write_json(cnv_deep_choice_doc, cnv_deep_choice_path)
print(f"Deep CNV imputer choice written: {cnv_deep_choice_path}")

global_prot_copy = IN_META / PROT_DEEP_LOCK_FILENAME
write_json(prot_deep_choice_doc, global_prot_copy)
print(f"Global proteomics deep-imputer copy written: {global_prot_copy}")

global_cnv_copy = IN_META / CNV_DEEP_LOCK_FILENAME
write_json(cnv_deep_choice_doc, global_cnv_copy)
print(f"Global CNV deep-imputer copy written: {global_cnv_copy}")

DEEP IMPUTATION BAKE-OFF FOR PROTEOMICS (3 seeds, leakage-safe)
[resume] Seeds remaining: [19537, 1584678, 17052356]
  Seed 19537
  prot_combined_union: GroupKFold(n_splits=10), 679 cells
    Building/loading deep fold caches for prot_combined_union...
    [mid-checkpoint] drug 5/100 | artifacts/reports/notebook 5_deep_imputation/deep_impute_bakeoff_seed19537_prot_combined_union.csv
    [mid-checkpoint] drug 10/100 | artifacts/reports/notebook 5_deep_imputation/deep_impute_bakeoff_seed19537_prot_combined_union.csv
    [mid-checkpoint] drug 15/100 | artifacts/reports/notebook 5_deep_imputation/deep_impute_bakeoff_seed19537_prot_combined_union.csv
    [mid-checkpoint] drug 20/100 | artifacts/reports/notebook 5_deep_imputation/deep_impute_bakeoff_seed19537_prot_combined_union.csv
    [mid-checkpoint] drug 25/100 | artifacts/reports/notebook 5_deep_imputation/deep_impute_bakeoff_seed19537_prot_combined_union.csv
    [mid-checkpoint] drug 30/100 | artifacts/reports/notebook 5_deep_imputatio

,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman,baseline_strategy,delta_vs_baseline
0,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,3,100,0.209616,0.225656,0.115639,-0.061630,0.209616,no_prot_reference,0.000000
1,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_aux_rna_cnv_mut::deep_dae,3,100,0.197535,0.212897,0.117882,-0.099805,0.196356,reference_without_prot,0.001179
2,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_aux_rna_cnv_mut::deep_dae+indicators,3,100,0.197535,0.212897,0.117882,-0.099805,0.196356,reference_without_prot,0.001179
3,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_dae,3,100,0.197522,0.212801,0.117855,-0.099809,0.196356,reference_without_prot,0.001166
4,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_dae+indicators,3,100,0.197522,0.212801,0.117855,-0.099809,0.196356,reference_without_prot,0.001166
5,2,prot_rppa_ccle,elasticnet,rna+prot,True,reference_without_prot,3,100,0.196356,0.208665,0.117851,-0.099308,0.196356,reference_without_prot,0.000000
6,3,prot_rppa_ccle,elasticnet,rna,False,no_prot_reference,3,100,0.196356,0.208665,0.117851,-0.099308,0.196356,no_prot_reference,0.000000
7,4,prot_combined_union,elasticnet,rna+mut,False,no_prot_reference,3,100,0.204024,0.215180,0.112710,-0.070314,0.204024,no_prot_reference,0.000000
8,5,prot_combined_union,elasticnet,rna+mut+prot,True,reference_without_prot,3,100,0.204024,0.215180,0.112710,-0.070314,0.204024,reference_without_prot,0.000000
9,5,prot_combined_union,elasticnet,rna+mut+prot,True,deep_dae,3,100,0.164851,0.171814,0.096221,-0.157191,0.204024,reference_without_prot,-0.039173



Per-seed deep proteomics summary:


,seed,config_rank,arm,model,feature_set,uses_prot,imputer_strategy,n_drugs,mean_spearman,median_spearman,std_spearman
0,19537,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.210062,0.227849,0.114269
33,1584678,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.209630,0.227267,0.117288
66,17052356,1,prot_combined_union,elasticnet,rna,False,no_prot_reference,100,0.209157,0.224963,0.115339
1,19537,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_aux_rna_cnv_mut::deep_dae,100,0.196334,0.206224,0.118630
2,19537,2,prot_rppa_ccle,elasticnet,rna+prot,True,deep_aux_rna_cnv_mut::deep_dae+indicators,100,0.196334,0.206224,0.118630
...,...,...,...,...,...,...,...,...,...,...,...
65,1584678,10,prot_procan_depmapSanger,elasticnet,prot,True,deep_dae+indicators,100,0.201044,0.197212,0.124527
96,17052356,10,prot_procan_depmapSanger,elasticnet,prot,True,deep_aux_rna_cnv_mut::deep_dae+indicators,100,0.207576,0.218831,0.119748
98,17052356,10,prot_procan_depmapSanger,elasticnet,prot,True,deep_dae+indicators,100,0.204675,0.210300,0.124308
95,17052356,10,prot_procan_depmapSanger,elasticnet,prot,True,deep_aux_rna_cnv_mut::deep_dae,100,0.201990,0.198168,0.121604


DEEP CNV IMPUTATION SENSITIVITY BAKE-OFF (3 seeds, leakage-safe)
  Seed 19537
  CNV/prot_combined_union: GroupKFold(n_splits=10), 679 cells
    CNV/prot_combined_union: drugs_run=100, drugs_skipped=0
  CNV/prot_procan_depmapSanger: GroupKFold(n_splits=10), 485 cells
    CNV/prot_procan_depmapSanger: drugs_run=100, drugs_skipped=0
  CNV/prot_rppa_ccle: GroupKFold(n_splits=10), 612 cells
    CNV/prot_rppa_ccle: drugs_run=100, drugs_skipped=0
  Seed 1584678
  CNV/prot_combined_union: GroupKFold(n_splits=10), 679 cells
    CNV/prot_combined_union: drugs_run=100, drugs_skipped=0
  CNV/prot_procan_depmapSanger: GroupKFold(n_splits=10), 485 cells
    CNV/prot_procan_depmapSanger: drugs_run=100, drugs_skipped=0
  CNV/prot_rppa_ccle: GroupKFold(n_splits=10), 612 cells
    CNV/prot_rppa_ccle: drugs_run=100, drugs_skipped=0
  Seed 17052356
  CNV/prot_combined_union: GroupKFold(n_splits=10), 679 cells
    CNV/prot_combined_union: drugs_run=100, drugs_skipped=0
  CNV/prot_procan_depmapSanger: Group

/tmp/ipykernel_9944/351051864.py:644: RuntimeWarning: Mean of empty slice
  spearman=("spearman", lambda x: float(np.nanmean(x))),



Deep CNV bake-off summary:


,config_rank,arm,model,feature_set,imputer_strategy,n_seeds,n_drugs,mean_spearman,median_spearman,std_spearman,mean_r2,baseline_mean_spearman,delta_vs_baseline
0,1000,prot_combined_union,elasticnet,cnv,cnv::deep_cnv_dae,3,100,0.103065,0.092465,0.088966,-0.208784,NaN,NaN
1,1000,prot_combined_union,elasticnet,cnv,cnv::deep_cnv_dae+indicators,3,100,0.103065,0.092465,0.088966,-0.208784,NaN,NaN
2,1000,prot_procan_depmapSanger,elasticnet,cnv,cnv::deep_cnv_dae,3,100,0.102648,0.107345,0.113308,-0.319116,NaN,NaN
3,1000,prot_procan_depmapSanger,elasticnet,cnv,cnv::deep_cnv_dae+indicators,3,100,0.102648,0.107345,0.113308,-0.319116,NaN,NaN
4,1000,prot_rppa_ccle,elasticnet,cnv,cnv::deep_cnv_dae,3,100,0.101539,0.095103,0.097339,-0.258326,NaN,NaN
5,1000,prot_rppa_ccle,elasticnet,cnv,cnv::deep_cnv_dae+indicators,3,100,0.101539,0.095103,0.097339,-0.258326,NaN,NaN
6,1001,prot_combined_union,elasticnet,rna+cnv,reference_without_cnv,3,100,0.209616,0.225656,0.115639,-0.061630,0.209616,0.000000
7,1001,prot_rppa_ccle,elasticnet,rna+cnv,reference_without_cnv,3,100,0.196356,0.208665,0.117851,-0.099308,0.196356,0.000000
8,1001,prot_procan_depmapSanger,elasticnet,rna+cnv,reference_without_cnv,3,100,0.187905,0.192800,0.121655,-0.134731,0.187905,0.000000
9,1001,prot_combined_union,elasticnet,rna+cnv,cnv::deep_cnv_dae,3,100,0.175256,0.176361,0.101159,-0.198231,0.209616,-0.034361


DEEP IMPUTER LOCK DECISION

rank1__prot_combined_union__elasticnet__rna:
  Chosen   : not_applicable

rank2__prot_rppa_ccle__elasticnet__rna+prot:
  Chosen   : deep_aux_rna_cnv_mut::deep_dae
  Spearman : 0.1975 ± 0.1179
  Baseline : 0.1964  delta=+0.0012

rank3__prot_rppa_ccle__elasticnet__rna:
  Chosen   : not_applicable

rank4__prot_combined_union__elasticnet__rna+mut:
  Chosen   : not_applicable

rank5__prot_combined_union__elasticnet__rna+mut+prot:
  Chosen   : deep_dae+indicators
  Spearman : 0.1636 ± 0.0963
  Baseline : 0.2040  delta=-0.0404

rank6__prot_combined_union__elasticnet__rna+prot:
  Chosen   : deep_dae+indicators
  Spearman : 0.1652 ± 0.1003
  Baseline : 0.2096  delta=-0.0445

rank7__prot_procan_depmapSanger__elasticnet__rna+mut+prot:
  Chosen   : deep_aux_rna_cnv_mut::deep_dae
  Spearman : 0.1892 ± 0.1146
  Baseline : 0.1790  delta=+0.0102

rank8__prot_rppa_ccle__elasticnet__rna+mut:
  Chosen   : not_applicable

rank9__prot_rppa_ccle__elasticnet__rna+mut+prot:
  Chose

# Notebook 5: Deep Imputation Analysis and Comparison with Notebook 4

## Purpose and analytical scope

Notebook 5 extends the imputation study from Notebook 4 by replacing simple fold-safe imputers with **deep denoising autoencoder-based imputers**, while preserving the same overall evaluation framework. The notebook is designed to answer a focused methodological question: **does deep imputation improve downstream drug response prediction, relative to the simpler imputation strategies evaluated previously?**

To ensure that the comparison remains meaningful, Notebook 5 keeps the key elements of the earlier design unchanged. It uses the same three random seeds, the same 100 PRISM AUC drugs selected by coverage, the same lineage-aware cross-validation procedure, the same arm-specific cohorts, and the same downstream ElasticNet evaluation model. In this sense, Notebook 5 is not a new predictive modelling notebook, but a more advanced **imputation sensitivity analysis** layered on top of the Notebook 4 framework.

The notebook evaluates two related but distinct tasks. First, it studies **deep proteomics imputation** for the ranked proteomics-containing configurations inherited from Notebook 4. Secondly, it studies **deep CNV imputation sensitivity** across the expanded CNV-focused feature combinations. In both cases, the central goal is to assess whether a learned latent reconstruction model provides a better representation of the missing modality than simpler approaches such as mean, median, or k-nearest neighbours imputation.


## What Notebook 5 does methodologically

For **proteomics**, Notebook 5 evaluates two deep strategies:

1. a **deep denoising autoencoder** that reconstructs the proteomics block from its own partially observed values, and
2. a **conditional deep autoencoder** that reconstructs proteomics using auxiliary information from RNA, CNV, and mutation embeddings from the same training fold.

This conditional variant is the deep analogue of the auxiliary proteomics imputation idea explored in Notebook 4. The important difference is that reconstruction is now learned through a neural network rather than through a shallow imputation rule.

For **CNV**, the notebook evaluates a deep denoising autoencoder that reconstructs the CNV block, again strictly within each fold. It then plugs this imputed CNV representation into the same feature combinations considered previously, including `cnv`, `rna+cnv`, `cnv+mut`, `rna+cnv+mut`, `cnv+prot`, `rna+cnv+prot`, and `rna+cnv+mut+prot`.

The notebook also preserves the leakage-safe structure of the earlier analysis. All standardisation, latent reconstruction, and PCA fitting are performed using training fold data only. This is an important design decision because it means the deep models are being evaluated under the same strict conditions as the simple imputers in Notebook 4.

A further practical feature of Notebook 5 is that it retains the caching and checkpointing logic established earlier. Intermediate fold-level representations are cached, arm-level outputs are checkpointed, and seed-level results are saved progressively. This makes the notebook reproducible and robust to long runtimes.

## Main findings from the deep proteomics bake-off

The deep proteomics results show that deep imputation is **not uniformly beneficial**. Its effect depends strongly on the proteomics arm.

### RPPA arm

For the RPPA arm, deep imputation produces only **very small improvements** over the no-proteomics reference. In rank 2 (`rna+prot`), the best method is `deep_aux_rna_cnv_mut::deep_dae`, which reaches a mean Spearman of **0.1975**, compared with a baseline of **0.1964**. The improvement is therefore only **+0.0012**. Rank 9 (`rna+mut+prot`) behaves similarly, improving from **0.1898** to **0.1901**, which is effectively negligible.

This suggests that RPPA can be incorporated without harming performance, but deep imputation does not materially change the predictive picture. The deep model is mostly recovering a representation that is only marginally more useful than simply omitting proteomics.

### ProCan arm

The ProCan arm provides the strongest evidence in favour of deep imputation. In rank 7 (`rna+mut+prot`), the baseline without proteomics is **0.1790**, while the best deep auxiliary method reaches **0.1892**, yielding an uplift of **+0.0102**. This is the clearest positive result in Notebook 5.

This is an important finding because it suggests that when proteomics is relatively informative and not dominated by severe structural platform missingness, a conditional deep model can exploit correlations between modalities to recover a more useful latent representation. In other words, deep imputation appears to be most helpful when the modality being reconstructed is biologically rich and the missingness pattern is still learnable.

Rank 10 (`prot` only) also performs reasonably well, with the best deep method reaching **0.2061**. This confirms that the ProCan proteomics arm retains meaningful standalone signal, although it is still slightly below the best result from Notebook 4.

### Combined union arm

The combined union arm remains the clearest failure case. In rank 5 (`rna+mut+prot`), performance drops from a baseline of **0.2040** to roughly **0.163 to 0.165** under deep imputation. In rank 6 (`rna+prot`), performance drops from **0.2096** to roughly **0.165 to 0.169**. These are large negative deltas of approximately **-0.04**.

This result is highly consistent with the missingness structure of the combined union arm. Missingness here is not simply random or feature-local. It is driven by **platform availability**, meaning that whole platform-specific blocks are absent for some cell lines. A deep autoencoder can denoise partially observed features, but it cannot reliably recover information that was never measured at the platform level. As a result, the learned reconstruction appears to blur or distort the signal rather than improve it.


## Main findings from the deep CNV bake-off

The deep CNV analysis is considerably less favourable overall.

### CNV alone

When evaluated on its own, CNV produces modest performance, with mean Spearman values around **0.101 to 0.103** across arms. This confirms that CNV is a relatively weak standalone modality for this task.

### CNV combined with stronger modalities

Once stronger modalities such as RNA or proteomics are present, deep-imputed CNV generally **reduces performance**. For example:

* `rna+cnv` drops from **0.2096** to **0.1753** in the combined union arm,
* `rna+cnv` drops from **0.1964** to **0.1704** in RPPA,
* `rna+cnv+mut` drops from **0.2040** to **0.1793** in the combined union arm,
* `rna+cnv+prot` drops from **0.1975** to **0.1718** in RPPA,
* `rna+cnv+mut+prot` drops from **0.1905** to **0.1631** in ProCan.

This indicates that deep CNV reconstruction is not adding complementary predictive information in the settings that matter most. Instead, it is introducing noise or over-smoothing the CNV signal, which weakens the downstream ElasticNet model.

### The `cnv+mut` exception

The one case in which deep CNV appears beneficial is `cnv+mut`. Here the reference model is essentially mutation-only, which performs poorly. Adding deep-imputed CNV raises performance substantially, for example from **0.0247** to **0.1080** in the combined union arm. However, this should not be over-interpreted. The improvement is real, but the absolute performance remains far below the RNA-based models. The result therefore shows that CNV carries more signal than mutation alone, not that deep CNV imputation is broadly optimal.

## Comparison with Notebook 4

Notebook 4 evaluated simpler imputers, specifically fold-safe **mean**, **median**, **k-nearest neighbours**, and indicator-enhanced variants, including auxiliary proteomics imputation with RNA, CNV, and mutation information. Notebook 5 asks whether replacing those simpler methods with deep denoising autoencoders changes the conclusion.

Overall, the answer is **no in general, and only weakly yes in a narrow subset of cases**.

### Proteomics comparison

A direct comparison of the best-performing strategies from the two notebooks shows that Notebook 5 does not improve the overall picture:

| Configuration                          | Notebook 4 best | Notebook 5 best |  Change |
| -------------------------------------- | --------------: | --------------: | ------: |
| Rank 2, RPPA, `rna+prot`               |          0.1996 |          0.1975 | -0.0021 |
| Rank 5, combined union, `rna+mut+prot` |          0.1793 |          0.1636 | -0.0157 |
| Rank 6, combined union, `rna+prot`     |          0.1822 |          0.1652 | -0.0170 |
| Rank 7, ProCan, `rna+mut+prot`         |          0.1903 |          0.1892 | -0.0011 |
| Rank 9, RPPA, `rna+mut+prot`           |          0.1921 |          0.1901 | -0.0020 |
| Rank 10, ProCan, `prot`                |          0.2123 |          0.2061 | -0.0062 |

These comparisons show that the deep methods do **not** outperform the simpler Notebook 4 methods in the configurations that matter most. Even in the ProCan arm, where deep auxiliary imputation is clearly useful relative to the no-proteomics reference, it still falls slightly short of the best Notebook 4 result.

The most important qualitative similarity between the two notebooks is that they agree on the **direction of the modality effect**:

* RPPA yields only tiny gains.
* ProCan can yield a modest positive uplift.
* Combined union proteomics hurts performance when added through imputation.

Thus, Notebook 5 largely confirms the substantive conclusions of Notebook 4 rather than overturning them.

### CNV comparison

The CNV comparison is even clearer. Notebook 4 already showed that imputed CNV was generally unhelpful whenever RNA was present, and Notebook 5 reproduces almost the same pattern.

For instance, in the combined union arm:

* Notebook 4 `rna+cnv`: **0.1768**
* Notebook 5 `rna+cnv`: **0.1753**

In the RPPA arm:

* Notebook 4 `rna+cnv`: **0.1715**
* Notebook 5 `rna+cnv`: **0.1704**

In the ProCan arm:

* Notebook 4 `rna+cnv`: **0.1500**
* Notebook 5 `rna+cnv`: **0.1492**

These differences are extremely small and all remain well below the no-CNV references. This indicates that deep CNV imputation does not meaningfully improve upon the simpler imputers from Notebook 4. In practice, the deep model is reproducing the same negative conclusion with more computational complexity.


## Important methodological note on baseline comparability

One subtle point should be acknowledged when comparing Notebook 5 to Notebook 4. The baseline values are not numerically identical in every case. For example, the RNA-only reference in Notebook 4 was approximately **0.2108**, whereas the corresponding Notebook 5 reference is **0.2096**. This likely reflects the fact that Notebook 5 uses its own fold-safe block transformation path for the base modalities, rather than reproducing the exact same preprocessing stack from Notebook 4.

This does not invalidate the comparison, but it does mean that the most important comparison is not the absolute baseline level itself, but rather the **relative effect of adding the imputed modality within each notebook**. On that criterion, the conclusion is stable: deep imputation does not provide a broad improvement over the simpler approaches from Notebook 4.


## Interpretation

The overall interpretation is that **deep imputation is not a universal upgrade** for this problem. Its utility depends on the missingness structure and the modality being reconstructed.

For proteomics, deep imputation is most convincing in the **ProCan arm**, especially when auxiliary RNA, CNV, and mutation information is provided. This suggests that deep conditional reconstruction can capture cross-modal structure when the target modality contains genuinely informative biology and the missingness is still learnable.

For the **combined union arm**, deep imputation performs poorly because the dominant issue is structural platform absence rather than feature-level noise. In such a setting, the autoencoder is attempting to reconstruct information that is systematically unobserved, which leads to degraded downstream performance.

For **CNV**, the deep autoencoder does not appear to solve an important bottleneck in the pipeline. CNV is not the principal missing modality, and its reconstructed representation seems to offer little benefit once stronger modalities are already available. The result is that deep CNV imputation is mostly redundant or harmful.


## Conclusion

Notebook 5 substantially extends the methodological depth of the imputation study by introducing denoising autoencoders and conditional auxiliary reconstruction, while preserving the same leakage-safe evaluation design established in Notebook 4. However, the empirical results show that this added modelling complexity does not translate into broad predictive gains.

Compared with Notebook 4, deep imputation offers at most **modest arm-specific benefits**, most notably for the ProCan proteomics arm, but it does not improve the overall pipeline and in several important settings performs worse than the simpler imputation strategies. The strongest negative result remains the combined union proteomics arm, where deep reconstruction substantially degrades performance. Deep CNV imputation is similarly unconvincing, yielding no practical advantage over the simpler alternatives.

Taken together, the results suggest that **Notebook 4 should remain the primary imputation reference**, with Notebook 5 serving as an important confirmatory and exploratory extension. Its main contribution is not that deep imputation becomes the new default, but that it clarifies where more complex imputers are useful, where they are neutral, and where they are clearly inappropriate.
